# Sobrino2004 — Emissividade TM6

Estima a emissividade da banda 6 do Landsat 5 TM. Pixels com NDVI < 0,2 ficam NaN nesta implementação; não há relação de solo inferida automaticamente.

## Precisa de

**Bandas ópticas — obrigatórias na forma de uso deste exemplo:**

- **Vermelho:** imagem calibrada em reflectância de superfície, **adimensional**.
- **Infravermelho próximo (NIR):** imagem calibrada na **mesma grandeza e unidade do vermelho**.

A função calcula o NDVI internamente. Como alternativa, fornecer somente `ndvi_image` já preparado, **adimensional, em [-1,1]**, no lugar das duas bandas; não combinar as entradas.

**Banda de destino:**

- **`band`:** número da banda, **sem unidade**; usar **6**, correspondente a Landsat 5 TM. Não atribuir esta emissividade a TIRS B10/B11.

**Entrada opcional de qualidade:**

- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar, sujeito às validações do método. Neste exemplo, sua origem é descrita na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

- **Vermelho superficial:** GeoTIFF com valores físicos de **reflectância de superfície, adimensionais**.
- **NIR superficial:** GeoTIFF com valores físicos de **reflectância de superfície, adimensionais**.

O leitor confere a grade e usa os NoData dos arquivos. Não monta automaticamente uma máscara de nuvens: complemente a máscara com a qualidade e as classes inadequadas para o método. Não é necessário fornecer MTL nem bandas térmicas para calcular somente a emissividade a partir dessas imagens prontas.

## Como as imagens são preparadas

Este notebook recebe vermelho e NIR **já em reflectância de superfície**, adimensional. Ele lê os GeoTIFFs e seus NoData, mas não converte DN em reflectância e não aplica correção atmosférica.

Ao usar um produto de reflectância de superfície com valores inteiros escalados, aplique antes o fator de escala e o offset documentados pelo fornecedor. Não usar os inteiros armazenados como se já fossem reflectância física.

Reflectância TOA (topo da atmosfera) e reflectância de superfície são grandezas diferentes. A conversão de DN para TOA, sozinha, não produz a reflectância superficial necessária neste exemplo.

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = emissivity(
    # Método
    method='sobrino-2004-tm6',

    # Banda de destino
    band=6,

    # Entradas para o NDVI
    red=vermelho,
    nir=nir,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "sobrino-2004-tm6" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None → 6` | Número da banda | Somente 6, Landsat5 TM; não atribuir a TIRS10/11. |
| `red` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | Reflectância superficial (adimensional) | Banda vermelha calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `nir` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | Reflectância superficial (adimensional) | Banda NIR calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — NDVI preparado

Usar NDVI já calculado e alinhado no lugar do par vermelho/NIR. Não há necessidade de chamar uma função pública de NDVI.

```python
resultado = emissivity(
    # Método
    method='sobrino-2004-tm6',

    # Banda de destino
    band=6,

    # Entradas para o NDVI
    ndvi_image=ndvi_preparado,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "sobrino-2004-tm6" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None → 6` | Número da banda | Somente 6, Landsat5 TM; não atribuir a TIRS10/11. |
| `ndvi_image` | Condicional: alternativa ao par vermelho/NIR | `None` | Adimensional, [-1,1] | NDVI já preparado e alinhado; omitir as duas bandas ópticas. A convenção radiométrica deve respeitar a fonte do método. Não combinar NDVI preparado e vermelho/NIR. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import emissivity
import rasterio

vermelho_path = Path("CAMINHO_DO_VERMELHO_REFLECTANCIA.tif")
nir_path = Path("CAMINHO_DO_NIR_REFLECTANCIA.tif")
saida = Path("resultados/sobrino_2004_tm6")
saida.mkdir(parents=True, exist_ok=True)

## Ler as imagens

As duas bandas precisam estar na mesma grade e ja em reflectancia. O exemplo nao transforma DN em reflectancia de superficie nem calcula correcao atmosferica. `rasterio.open` le os arquivos e fornece o perfil de georreferenciamento.

### Máscara usada neste exemplo

Os arquivos de reflectância são lidos com seus NoData. A máscara booleana reúne pixels não finitos; True exclui. Complemente-a com nuvens, sombras, água/neve e classes incompatíveis com o modelo usando a informação de qualidade dos seus produtos. O leitor destes dois rasters não cria QA de nuvens automaticamente.


In [ ]:
with rasterio.open(vermelho_path) as src:
    vermelho = src.read(1, masked=True).astype(float).filled(np.nan)
    perfil = src.profile.copy()
    grade = (src.shape, src.crs, src.transform)
with rasterio.open(nir_path) as src:
    if (src.shape, src.crs, src.transform) != grade:
        raise ValueError("As bandas vermelho/NIR precisam estar alinhadas.")
    nir = src.read(1, masked=True).astype(float).filled(np.nan)
mascara = ~np.isfinite(vermelho) | ~np.isfinite(nir)

## Aplicar o metodo

A mascara tambem deve excluir agua/neve e superficies inadequadas; NDVI sozinho nao identifica essas classes.

In [ ]:
imagem = emissivity(
    # Método
    method='sobrino-2004-tm6',

    # Banda de destino
    band=6,

    # Entradas para o NDVI
    red=vermelho,
    nir=nir,

    # Exclusão de pixels
    mask=mascara,
)


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
perfil.update(dtype="float32", count=1, nodata=np.nan)
with rasterio.open(saida / "emissividade.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units="dimensionless")
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

[Artigo ou documentacao original](https://doi.org/10.1016/j.rse.2004.02.003).

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).